# FTIR-Based TBN Model — Parameter Fitting

Fits and validates the **Model C** (5-band MLR) and **Model D** (peak-wavenumber MLR) 
from Wolak et al. *Energies* 2022, 15, 2809 using the paper's Table 5 dataset.

Then builds an **extended formula** that estimates absorbance (and therefore TBN) 
as a function of wavenumber, temperature, turbidity, and fuel dilution.

Outputs `ftir_model.npz` — copy this to the Raspberry Pi alongside `centroids.bin`.

In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
import os, importlib.util, sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

WORK = '/kaggle/working'
os.makedirs(f'{WORK}/models', exist_ok=True)
os.makedirs(f'{WORK}/data',   exist_ok=True)

# Load pipeline
PIPELINE_PATH = '/kaggle/input/datasets/krishnasimha/daq-pipeline-ftir/daq_pipeline_ftir (1).py'  # adjust to your dataset name
spec = importlib.util.spec_from_file_location('daq', PIPELINE_PATH)
daq  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(daq)

daq.MODEL_PATH      = f'{WORK}/models/centroids.bin'
daq.INTERP_PATH     = f'{WORK}/models/interpolation.npz'
daq.FTIR_MODEL_PATH = f'{WORK}/models/ftir_model.npz'
daq.DATA_LOG_PATH   = f'{WORK}/data/readings.csv'
daq.LAB_DATA_PATH   = f'{WORK}/data/lab_samples.csv'

print('Pipeline loaded. Paths patched to /kaggle/working')

Pipeline loaded. Paths patched to /kaggle/working


## Step 1 — Paper Dataset (Table 5, Wolak 2022)

13 oil samples from a BMW 520d driven 14,820 km total.  
Each row: 5 averaged differential absorbance values + lab-measured TBN.

In [2]:
# ── Cell 2: Raw dataset (Table 5, Model C) ────────────────────────────────
# Columns: abs_lt1000, abs_1000_1450, abs_1475_1800, abs_1800_2830, abs_2975_4000, TBN

BAND_NAMES = ['abs_lt1000', 'abs_1000_1450', 'abs_1475_1800', 'abs_1800_2830', 'abs_2975_4000']
SAMPLE_IDS = ['SMP1','SMP2','SMP3','SMP4','SMP5','SMP7',
               'SMP8','SMP9','SMP10','SMP11','SMP12','SMP13','SMP14']

raw = np.array([
    # abs<1000  abs1000-1450  abs1475-1800  abs1800-2830  abs2975-4000  TBN
    [0.0281,    0.0598,       0.0627,       0.0524,       0.0771,       8.18],  # SMP1
    [0.0417,    0.0960,       0.0996,       0.0800,       0.1196,       6.13],  # SMP2
    [0.0508,    0.1177,       0.1210,       0.1010,       0.1524,       5.24],  # SMP3
    [0.0629,    0.1432,       0.1450,       0.1243,       0.1870,       5.07],  # SMP4
    [0.0824,    0.1855,       0.1829,       0.1582,       0.2371,       4.67],  # SMP5
    [0.1100,    0.2400,       0.2377,       0.2102,       0.3152,       4.36],  # SMP7
    [0.1227,    0.2666,       0.2648,       0.2377,       0.3583,       2.85],  # SMP8
    [0.1361,    0.2944,       0.2932,       0.2629,       0.3979,       3.10],  # SMP9
    [0.1587,    0.3289,       0.3363,       0.3007,       0.4507,       2.13],  # SMP10
    [0.1688,    0.3497,       0.3618,       0.3196,       0.4804,       1.92],  # SMP11
    [0.1982,    0.3983,       0.4176,       0.3736,       0.5583,       1.67],  # SMP12
    [0.2138,    0.4265,       0.4489,       0.4022,       0.6004,       0.58],  # SMP13
    [0.2302,    0.4563,       0.4825,       0.4375,       0.6503,       0.50],  # SMP14
], dtype=np.float64)

X_bands = raw[:, :5]    # 5 absorbance band averages
y_tbn   = raw[:, 5]     # measured TBN

df = pd.DataFrame(raw, columns=BAND_NAMES + ['TBN_measured'], index=SAMPLE_IDS)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Paper dataset (Table 5):')
print(df)

Paper dataset (Table 5):
       abs_lt1000  abs_1000_1450  abs_1475_1800  abs_1800_2830  abs_2975_4000  \
SMP1       0.0281         0.0598         0.0627         0.0524         0.0771   
SMP2       0.0417         0.0960         0.0996         0.0800         0.1196   
SMP3       0.0508         0.1177         0.1210         0.1010         0.1524   
SMP4       0.0629         0.1432         0.1450         0.1243         0.1870   
SMP5       0.0824         0.1855         0.1829         0.1582         0.2371   
SMP7       0.1100         0.2400         0.2377         0.2102         0.3152   
SMP8       0.1227         0.2666         0.2648         0.2377         0.3583   
SMP9       0.1361         0.2944         0.2932         0.2629         0.3979   
SMP10      0.1587         0.3289         0.3363         0.3007         0.4507   
SMP11      0.1688         0.3497         0.3618         0.3196         0.4804   
SMP12      0.1982         0.3983         0.4176         0.3736         0.5583   
SMP

## Step 2 — Fit Model C (Best Model: R²=0.982)

**Formula (Model C):**
```
TBN = 9.682
    + 316.270 × Abs(<1000 cm⁻¹)
    -  10.326 × Abs(1000–1450 cm⁻¹)
    - 133.000 × Abs(1475–1800 cm⁻¹)
    -  36.081 × Abs(1800–2830 cm⁻¹)
    +   3.841 × Abs(2975–4000 cm⁻¹)
```

Each `Abs(band)` = arithmetic mean of differential absorbance values 
across all wavenumbers within that spectral range.

In [3]:
# ── Cell 3: Fit Model C from scratch and verify against paper ─────────────
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

reg_c = LinearRegression().fit(X_bands, y_tbn)
y_pred_c = reg_c.predict(X_bands)

r2_c   = r2_score(y_tbn, y_pred_c)
rmse_c = np.sqrt(mean_squared_error(y_tbn, y_pred_c))

print('── Model C Coefficients ──────────────────────────────────')
print(f'  Intercept       : {reg_c.intercept_:>10.4f}   (paper: 9.682)')
for name, coef in zip(BAND_NAMES, reg_c.coef_):
    paper = {'abs_lt1000':316.270,'abs_1000_1450':-10.326,
             'abs_1475_1800':-133.000,'abs_1800_2830':-36.081,'abs_2975_4000':3.841}[name]
    print(f'  {name:<20}: {coef:>10.4f}   (paper: {paper})')

print(f'\n  R²            : {r2_c:.4f}   (paper: 0.9815)')
print(f'  RMSE          : {rmse_c:.4f}   (paper: 0.4050)')

print('\n── Measured vs Predicted ─────────────────────────────────')
for sid, meas, pred in zip(SAMPLE_IDS, y_tbn, y_pred_c):
    diff = abs(meas - pred)
    bar = '█' * int(pred) + '░' * (9 - int(pred))
    print(f'  {sid:<6} measured={meas:.2f}  predicted={pred:.2f}  err={diff:.2f}  [{bar}]')

── Model C Coefficients ──────────────────────────────────
  Intercept       :     9.7109   (paper: 9.682)
  abs_lt1000          :   318.8579   (paper: 316.27)
  abs_1000_1450       :   -12.1408   (paper: -10.326)
  abs_1475_1800       :  -133.2630   (paper: -133.0)
  abs_1800_2830       :   -44.9917   (paper: -36.081)
  abs_2975_4000       :    10.3365   (paper: 3.841)

  R²            : 0.9817   (paper: 0.9815)
  RMSE          : 0.2961   (paper: 0.4050)

── Measured vs Predicted ─────────────────────────────────
  SMP1   measured=8.18  predicted=8.03  err=0.15  [████████░]
  SMP2   measured=6.13  predicted=6.21  err=0.08  [██████░░░]
  SMP3   measured=5.24  predicted=5.39  err=0.15  [█████░░░░]
  SMP4   measured=5.07  predicted=5.05  err=0.02  [█████░░░░]
  SMP5   measured=4.67  predicted=4.69  err=0.02  [████░░░░░]
  SMP7   measured=4.36  predicted=4.00  err=0.36  [███░░░░░░]
  SMP8   measured=2.85  predicted=3.32  err=0.47  [███░░░░░░]
  SMP9   measured=3.10  predicted=2.75  err=0.

In [4]:
# ── Cell 4: Fit Model D (peak wavenumbers) ────────────────────────────────
# Model D uses peak absorbance at 5 specific wavenumbers instead of band averages
# These correspond to specific molecular vibrations tied to TBN chemistry

PEAK_NAMES = ['abs_1746', 'abs_1631', 'abs_1196', 'abs_1169', 'abs_1062']

# Table 7 from paper
raw_d = np.array([
    # 1746cm  1631cm  1196cm  1169cm  1062cm  TBN
    [0.1314, 0.1090, 0.0830, 0.0869, 0.0575, 8.18],
    [0.2766, 0.1589, 0.1466, 0.1532, 0.0902, 6.13],
    [0.3400, 0.1871, 0.1801, 0.1863, 0.1114, 5.24],
    [0.4183, 0.2180, 0.2206, 0.2269, 0.1367, 5.07],
    [0.5922, 0.2680, 0.2924, 0.3135, 0.1748, 4.67],
    [0.7019, 0.3461, 0.3595, 0.3883, 0.2231, 4.36],
    [0.7274, 0.3859, 0.3877, 0.4180, 0.2448, 2.85],
    [0.7588, 0.4338, 0.4193, 0.4537, 0.2672, 3.10],
    [0.8730, 0.4981, 0.4745, 0.5163, 0.2994, 2.13],
    [0.9591, 0.5381, 0.5099, 0.5566, 0.3162, 1.92],
    [1.0948, 0.6098, 0.5774, 0.6275, 0.3590, 1.67],
    [1.1452, 0.6605, 0.6120, 0.6652, 0.3829, 0.58],
    [1.1873, 0.7054, 0.6440, 0.6968, 0.4087, 0.50],
], dtype=np.float64)

X_peaks = raw_d[:, :5]
y_tbn_d = raw_d[:, 5]   # same TBN values

reg_d = LinearRegression().fit(X_peaks, y_tbn_d)
y_pred_d = reg_d.predict(X_peaks)
r2_d   = r2_score(y_tbn_d, y_pred_d)
rmse_d = np.sqrt(mean_squared_error(y_tbn_d, y_pred_d))

print('── Model D Coefficients (peak wavenumbers) ───────────────')
print(f'  Intercept  : {reg_d.intercept_:>10.4f}   (paper: 9.033)')
paper_d = {'abs_1746':38.416,'abs_1631':14.971,'abs_1196':-287.984,
           'abs_1169':82.665,'abs_1062':154.500}
for name, coef in zip(PEAK_NAMES, reg_d.coef_):
    print(f'  {name:<12}: {coef:>10.4f}   (paper: {paper_d[name]})')
print(f'\n  R²   : {r2_d:.4f}   (paper: 0.9791)')
print(f'  RMSE : {rmse_d:.4f}   (paper: 0.4313)')

── Model D Coefficients (peak wavenumbers) ───────────────
  Intercept  :     9.0302   (paper: 9.033)
  abs_1746    :    39.0918   (paper: 38.416)
  abs_1631    :    14.7809   (paper: 14.971)
  abs_1196    :  -291.1182   (paper: -287.984)
  abs_1169    :    82.7023   (paper: 82.665)
  abs_1062    :   157.7384   (paper: 154.5)

  R²   : 0.9791   (paper: 0.9791)
  RMSE : 0.3162   (paper: 0.4313)


## Step 3 — Extended Formula: TBN as f(wavenumber, temp, turbidity, fuel dilution)

The paper only uses averaged absorbance bands. But you can build a secondary model that **estimates per-band absorbance** from physical conditions when a full spectrometer isn't available.

### Physical basis

| Predictor | Effect on absorbance |
|---|---|
| **Wavenumber** | Selects which molecular bond vibration is being measured. Different bands track different degradation products |
| **Temperature** | Shifts OH-stretch and carbonyl bands (1475–1800 cm⁻¹). Higher temp → more oxidation → higher absorbance in this band |
| **Turbidity (NTU)** | Proxy for soot + particulate load. Increases absorbance in baseline-raise bands (1800–2830, 2975–4000 cm⁻¹) |
| **Fuel dilution (%)** | Fuel-in-oil adds C-H stretch absorbers (2837–2979 cm⁻¹) and shifts fingerprint region |

### Extended MLR formula

```
Absorbance(band) ≈ α₀ + α₁·T + α₂·NTU + α₃·FD

TBN = β₀
    + β₁ · [α₀⁽¹⁾ + α₁⁽¹⁾·T + α₂⁽¹⁾·NTU + α₃⁽¹⁾·FD]      ← band <1000 cm⁻¹
    + β₂ · [α₀⁽²⁾ + α₁⁽²⁾·T + α₂⁽²⁾·NTU + α₃⁽²⁾·FD]      ← band 1000–1450 cm⁻¹
    + β₃ · [α₀⁽³⁾ + α₁⁽³⁾·T + α₂⁽³⁾·NTU + α₃⁽³⁾·FD]      ← band 1475–1800 cm⁻¹
    + β₄ · [α₀⁽⁴⁾ + α₁⁽⁴⁾·T + α₂⁽⁴⁾·NTU + α₃⁽⁴⁾·FD]      ← band 1800–2830 cm⁻¹
    + β₅ · [α₀⁽⁵⁾ + α₁⁽⁵⁾·T + α₂⁽⁵⁾·NTU + α₃⁽⁵⁾·FD]      ← band 2975–4000 cm⁻¹

Substituting and collecting terms:

TBN ≈ c₀ + c_T·T + c_NTU·NTU + c_FD·FD

where:
  c₀   = β₀ + Σ(βᵢ · α₀⁽ⁱ⁾)     intercept
  c_T  = Σ(βᵢ · α₁⁽ⁱ⁾)           temperature coefficient
  c_NTU= Σ(βᵢ · α₂⁽ⁱ⁾)           turbidity coefficient
  c_FD = Σ(βᵢ · α₃⁽ⁱ⁾)           fuel dilution coefficient
```

The α coefficients for each band are fitted from calibration data (or derived from the literature physics below). We use **physically derived priors** from the paper for the correction magnitudes.

In [5]:
# ── Cell 5: Extended formula — physical correction coefficients ───────────
# We cannot fit α from the paper data (no temp/turbidity/fuel-dil columns).
# Instead we derive physically motivated priors from the literature and
# from the paper's chemistry discussion, then express the combined formula.
#
# These are CALIBRATION PRIORS — update them with your own sensor data
# once you have paired (temp, turbidity, fuel_dil, absorbance) measurements.

# β coefficients from Model C (fitted above)
beta = np.array([reg_c.intercept_] + list(reg_c.coef_))
beta_labels = ['intercept'] + BAND_NAMES

# α: correction per band per physical variable
# Shape: (5 bands, 3 covariates: temp, turbidity, fuel_dil)
# Signs and magnitudes derived from paper physics discussion:
#   - temp: positive effect on carbonyl/nitro band (1475-1800), small elsewhere
#   - turbidity: positive effect on baseline-raise bands (1800-2830, 2975-4000)
#   - fuel_dil: dilutes all bands (negative correction) + adds CH stretch noise
#
# Units: absorbance-per-unit-of-covariate
#   temp in °C above 25°C (reference), turbidity in NTU/100, fuel_dil in %

alpha = np.array([
    # band          dAbs/dT    dAbs/dNTU   dAbs/dFD
    # abs_lt1000
    [              0.0002,     0.0050,     -0.0010],
    # abs_1000_1450
    [              0.0003,     0.0030,     -0.0015],
    # abs_1475_1800  ← most temperature-sensitive (carbonyl / nitro)
    [              0.0008,     0.0020,     -0.0010],
    # abs_1800_2830  ← most turbidity-sensitive (soot / resin baseline raise)
    [              0.0002,     0.0120,     -0.0005],
    # abs_2975_4000  ← fuel dilution adds CH-stretch here
    [              0.0001,     0.0080,      0.0030],
])

# Collapsed coefficients: TBN ≈ c0 + c_T·T + c_NTU·NTU + c_FD·FD
beta_bands = reg_c.coef_   # β₁..β₅
c_T   = float(np.dot(beta_bands, alpha[:, 0]))
c_NTU = float(np.dot(beta_bands, alpha[:, 1]))
c_FD  = float(np.dot(beta_bands, alpha[:, 2]))
c0    = float(reg_c.intercept_)

print('── Extended TBN Formula ──────────────────────────────────')
print()
print('  TBN = c₀ + c_T·(T − 25) + c_NTU·(NTU/100) + c_FD·FD%')
print()
print(f'  c₀   (intercept at reference conditions) = {c0:.4f} mg KOH/g')
print(f'  c_T  (per °C above 25°C)                 = {c_T:.5f} mg KOH/g per °C')
print(f'  c_NTU(per 100 NTU turbidity)              = {c_NTU:.4f} mg KOH/g per 100 NTU')
print(f'  c_FD (per 1% fuel-in-oil dilution)        = {c_FD:.5f} mg KOH/g per %')
print()
print('  Where T   = oil temperature (°C)')
print('        NTU = turbidity reading from optical sensor')
print('        FD% = fuel dilution fraction (%)  from sensor or estimated')
print()
print('  NOTE: c₀ here is the intercept when ALL absorbance bands are at their')
print('  reference-condition values (T=25°C, NTU=0, FD=0%).')
print('  The full Model C formula (direct from absorbances) remains the')
print('  primary path — this extended formula is a fallback when the FTIR')
print('  spectrometer output is unavailable or incomplete.')
print()
print('── Physical interpretation ───────────────────────────────')
print(f'  Every +10°C above reference reduces TBN by {abs(c_T*10):.3f} mg KOH/g')
print(f'  Every +100 NTU turbidity reduces TBN by    {abs(c_NTU):.3f} mg KOH/g')
print(f'  Every +1% fuel dilution changes TBN by     {c_FD:.4f} mg KOH/g')

── Extended TBN Formula ──────────────────────────────────

  TBN = c₀ + c_T·(T − 25) + c_NTU·(NTU/100) + c_FD·FD%

  c₀   (intercept at reference conditions) = 9.7109 mg KOH/g
  c_T  (per °C above 25°C)                 = -0.05445 mg KOH/g per °C
  c_NTU(per 100 NTU turbidity)              = 0.8341 mg KOH/g per 100 NTU
  c_FD (per 1% fuel-in-oil dilution)        = -0.11388 mg KOH/g per %

  Where T   = oil temperature (°C)
        NTU = turbidity reading from optical sensor
        FD% = fuel dilution fraction (%)  from sensor or estimated

  NOTE: c₀ here is the intercept when ALL absorbance bands are at their
  reference-condition values (T=25°C, NTU=0, FD=0%).
  The full Model C formula (direct from absorbances) remains the
  primary path — this extended formula is a fallback when the FTIR
  spectrometer output is unavailable or incomplete.

── Physical interpretation ───────────────────────────────
  Every +10°C above reference reduces TBN by 0.544 mg KOH/g
  Every +100 NTU turbidi

In [6]:
# ── Cell 6: Absorbance from wavenumber — band-membership model ────────────
#
# Given a raw spectrometer reading at a specific wavenumber, this function
# returns which Model C band it belongs to and the corrected absorbance.
#
# Abs_corrected(ν, T, NTU, FD) = Abs_raw(ν) + dA_T(ν)·(T-25) + dA_NTU(ν)·(NTU/100) + dA_FD(ν)·FD

BAND_RANGES = [
    ('abs_lt1000',    650,  1000),
    ('abs_1000_1450', 1000, 1450),
    ('abs_1475_1800', 1475, 1800),
    ('abs_1800_2830', 1800, 2830),
    ('abs_2975_4000', 2975, 4000),
]
NOISE_RANGES = [(1451, 1474), (2837, 2979)]

def wavenumber_to_band(nu: float) -> str | None:
    """Map a wavenumber (cm⁻¹) to its Model C band name. Returns None if in noise range."""
    for lo, hi in NOISE_RANGES:
        if lo <= nu <= hi:
            return None   # artifact range — excluded from averaging
    for name, lo, hi in BAND_RANGES:
        if lo <= nu < hi:
            return name
    return None

def absorbance_correction(nu: float, T: float = 25.0,
                           turbidity: float = 0.0,
                           fuel_dil: float = 0.0) -> float:
    """
    Additive correction to raw absorbance at wavenumber nu.
    Returns the total correction (add to raw Abs to get condition-corrected value).

    Args:
        nu        : wavenumber in cm⁻¹
        T         : oil temperature in °C
        turbidity : turbidity in NTU
        fuel_dil  : fuel-in-oil dilution in %

    Returns: float correction (may be negative)
    """
    band = wavenumber_to_band(nu)
    if band is None:
        return 0.0   # noise range — no correction applied
    band_idx = [b[0] for b in BAND_RANGES].index(band)
    dT   = T - 25.0
    dNTU = turbidity / 100.0
    correction = (alpha[band_idx, 0] * dT +
                  alpha[band_idx, 1] * dNTU +
                  alpha[band_idx, 2] * fuel_dil)
    return correction

def tbn_from_bands(band_abs: dict, T: float = 25.0,
                   turbidity: float = 0.0, fuel_dil: float = 0.0) -> float:
    """
    Predict TBN from a dict of per-band average absorbances, with
    optional temperature, turbidity, and fuel-dilution corrections.

    Args:
        band_abs  : {band_name: average_differential_absorbance}
        T, turbidity, fuel_dil: physical conditions

    Returns: TBN in mg KOH/g
    """
    coefs_c = {
        'intercept':      reg_c.intercept_,
        'abs_lt1000':     reg_c.coef_[0],
        'abs_1000_1450':  reg_c.coef_[1],
        'abs_1475_1800':  reg_c.coef_[2],
        'abs_1800_2830':  reg_c.coef_[3],
        'abs_2975_4000':  reg_c.coef_[4],
    }
    tbn = coefs_c['intercept']
    for i, name in enumerate(BAND_NAMES):
        raw_abs = band_abs.get(name, 0.0)
        band_idx = i
        dT   = T - 25.0
        dNTU = turbidity / 100.0
        corr = (alpha[band_idx, 0] * dT +
                alpha[band_idx, 1] * dNTU +
                alpha[band_idx, 2] * fuel_dil)
        tbn += coefs_c[name] * (raw_abs + corr)
    return float(np.clip(tbn, 0.0, 12.0))   # physical bounds

# ── Sanity check with SMP1 (fresh oil, T=25, no turbidity/dilution) ──────
smp1_bands = dict(zip(BAND_NAMES, X_bands[0]))
tbn_check  = tbn_from_bands(smp1_bands)
print(f'SMP1 TBN (measured=8.18, predicted): {tbn_check:.3f} mg KOH/g')

# ── Effect of temperature ─────────────────────────────────────────────────
print('\n── Effect of temperature on TBN estimate (SMP5 mid-life) ──')
smp5_bands = dict(zip(BAND_NAMES, X_bands[4]))
for temp in [20, 25, 40, 60, 80, 100]:
    tbn_t = tbn_from_bands(smp5_bands, T=temp)
    print(f'  T={temp:3d}°C → TBN={tbn_t:.3f}  (Δ from 25°C: {tbn_t - tbn_from_bands(smp5_bands):+.3f})')

print('\n── Effect of turbidity (NTU) at T=25°C ─────────────────────')
for ntu in [0, 50, 100, 200, 500]:
    tbn_ntu = tbn_from_bands(smp5_bands, turbidity=ntu)
    print(f'  NTU={ntu:4d} → TBN={tbn_ntu:.3f}  (Δ: {tbn_ntu - tbn_from_bands(smp5_bands):+.3f})')

print('\n── Effect of fuel dilution (%) at T=25°C ───────────────────')
for fd in [0, 1, 2, 5, 10]:
    tbn_fd = tbn_from_bands(smp5_bands, fuel_dil=fd)
    print(f'  FD={fd:3d}%  → TBN={tbn_fd:.3f}  (Δ: {tbn_fd - tbn_from_bands(smp5_bands):+.3f})')

SMP1 TBN (measured=8.18, predicted): 8.029 mg KOH/g

── Effect of temperature on TBN estimate (SMP5 mid-life) ──
  T= 20°C → TBN=4.964  (Δ from 25°C: +0.272)
  T= 25°C → TBN=4.692  (Δ from 25°C: +0.000)
  T= 40°C → TBN=3.875  (Δ from 25°C: -0.817)
  T= 60°C → TBN=2.786  (Δ from 25°C: -1.906)
  T= 80°C → TBN=1.697  (Δ from 25°C: -2.995)
  T=100°C → TBN=0.609  (Δ from 25°C: -4.083)

── Effect of turbidity (NTU) at T=25°C ─────────────────────
  NTU=   0 → TBN=4.692  (Δ: +0.000)
  NTU=  50 → TBN=5.109  (Δ: +0.417)
  NTU= 100 → TBN=5.526  (Δ: +0.834)
  NTU= 200 → TBN=6.360  (Δ: +1.668)
  NTU= 500 → TBN=8.863  (Δ: +4.171)

── Effect of fuel dilution (%) at T=25°C ───────────────────
  FD=  0%  → TBN=4.692  (Δ: +0.000)
  FD=  1%  → TBN=4.578  (Δ: -0.114)
  FD=  2%  → TBN=4.464  (Δ: -0.228)
  FD=  5%  → TBN=4.123  (Δ: -0.569)
  FD= 10%  → TBN=3.553  (Δ: -1.139)


In [7]:
# ── Cell 7: Save ftir_model.npz ───────────────────────────────────────────
# Saves all coefficients so the pipeline can load them on startup
# without needing sklearn installed on the RPi.

ftir_path = f'{WORK}/models/ftir_model.npz'

np.savez(
    ftir_path,
    # Model C: 5-band MLR
    model_c_intercept = np.array([reg_c.intercept_]),
    model_c_coef      = reg_c.coef_,                  # shape (5,)
    model_c_r2        = np.array([r2_c]),
    model_c_rmse      = np.array([rmse_c]),

    # Model D: peak-wavenumber MLR
    model_d_intercept = np.array([reg_d.intercept_]),
    model_d_coef      = reg_d.coef_,                  # shape (5,)
    model_d_r2        = np.array([r2_d]),
    model_d_rmse      = np.array([rmse_d]),

    # Extended formula: physical correction coefficients
    # alpha[band_idx, 0] = dAbs/dT
    # alpha[band_idx, 1] = dAbs/d(NTU/100)
    # alpha[band_idx, 2] = dAbs/dFD%
    alpha_corrections = alpha,

    # Collapsed extended coefficients for direct use without band intermediary
    extended_c0   = np.array([c0]),
    extended_c_T  = np.array([c_T]),
    extended_c_NTU= np.array([c_NTU]),
    extended_c_FD = np.array([c_FD]),

    # Band definitions for runtime use
    band_lo = np.array([r[1] for r in BAND_RANGES]),
    band_hi = np.array([r[2] for r in BAND_RANGES]),
)

size = os.path.getsize(ftir_path)
print(f'Saved ftir_model.npz → {ftir_path}  ({size} bytes)')

# Verify round-trip load
check = np.load(ftir_path)
assert abs(check['model_c_intercept'][0] - reg_c.intercept_) < 1e-6, 'Intercept mismatch'
assert check['alpha_corrections'].shape == (5, 3), 'Alpha shape wrong'
print('Round-trip load: OK')

Saved ftir_model.npz → /kaggle/working/models/ftir_model.npz  (4228 bytes)
Round-trip load: OK


In [8]:
# ── Cell 8: Full formula summary ──────────────────────────────────────────
print('='*65)
print('  FULL FORMULA SUMMARY')
print('='*65)
print()
print('── PRIMARY (Model C) — use when FTIR bands available ──────')
print()
print('  TBN = 9.682')
print(f'      + {reg_c.coef_[0]:>8.3f} × Abs(<1000 cm⁻¹)      # fingerprint region')
print(f'      + {reg_c.coef_[1]:>8.3f} × Abs(1000–1450 cm⁻¹)  # sulfone additives')
print(f'      + {reg_c.coef_[2]:>8.3f} × Abs(1475–1800 cm⁻¹)  # carbonyl/nitro products')
print(f'      + {reg_c.coef_[3]:>8.3f} × Abs(1800–2830 cm⁻¹)  # baseline soot/resin')
print(f'      + {reg_c.coef_[4]:>8.3f} × Abs(2975–4000 cm⁻¹)  # oxidation baseline')
print()
print(f'  R²={r2_c:.4f}  RMSE={rmse_c:.4f} mg KOH/g')
print()
print('── WITH CORRECTIONS — add to each band absorbance first ───')
print()
print('  Abs_corrected(band, T, NTU, FD) =')
print('    Abs_raw(band)')
print('    + α_T(band)   × (T − 25)       [°C above reference]')
print('    + α_NTU(band) × (NTU / 100)    [per 100 NTU turbidity]')
print('    + α_FD(band)  × FD%             [fuel dilution %]')
print()
print('  α values per band:')
for i, (name, _, _) in enumerate(BAND_RANGES):
    print(f'  {name:<20}  α_T={alpha[i,0]:.4f}  α_NTU={alpha[i,1]:.4f}  α_FD={alpha[i,2]:+.4f}')
print()
print('── FALLBACK (no spectrometer) — conditions only ────────────')
print()
print('  TBN ≈ c₀  +  c_T·(T−25)  +  c_NTU·(NTU/100)  +  c_FD·FD%')
print()
print(f'  c₀    = {c0:.4f}  mg KOH/g  (intercept)')
print(f'  c_T   = {c_T:.6f}  mg KOH/g per °C')
print(f'  c_NTU = {c_NTU:.4f}  mg KOH/g per 100 NTU')
print(f'  c_FD  = {c_FD:.6f}  mg KOH/g per % fuel dilution')
print()
print('  IMPORTANT: α corrections are PRIORS from paper chemistry.')
print('  Calibrate them with your own paired sensor measurements.')
print('  Run OfflineTrainer after collecting 20+ paired observations.')
print('='*65)

  FULL FORMULA SUMMARY

── PRIMARY (Model C) — use when FTIR bands available ──────

  TBN = 9.682
      +  318.858 × Abs(<1000 cm⁻¹)      # fingerprint region
      +  -12.141 × Abs(1000–1450 cm⁻¹)  # sulfone additives
      + -133.263 × Abs(1475–1800 cm⁻¹)  # carbonyl/nitro products
      +  -44.992 × Abs(1800–2830 cm⁻¹)  # baseline soot/resin
      +   10.337 × Abs(2975–4000 cm⁻¹)  # oxidation baseline

  R²=0.9817  RMSE=0.2961 mg KOH/g

── WITH CORRECTIONS — add to each band absorbance first ───

  Abs_corrected(band, T, NTU, FD) =
    Abs_raw(band)
    + α_T(band)   × (T − 25)       [°C above reference]
    + α_NTU(band) × (NTU / 100)    [per 100 NTU turbidity]
    + α_FD(band)  × FD%             [fuel dilution %]

  α values per band:
  abs_lt1000            α_T=0.0002  α_NTU=0.0050  α_FD=-0.0010
  abs_1000_1450         α_T=0.0003  α_NTU=0.0030  α_FD=-0.0015
  abs_1475_1800         α_T=0.0008  α_NTU=0.0020  α_FD=-0.0010
  abs_1800_2830         α_T=0.0002  α_NTU=0.0120  α_FD=-0.00

In [9]:
# ── Cell 9: Confirm all output files ─────────────────────────────────────
for fpath in [
    f'{WORK}/models/ftir_model.npz',
]:
    size = os.path.getsize(fpath)
    print(f'  {os.path.basename(fpath):<25} {size:>6} bytes  ✓')

print()
print('Copy to Raspberry Pi:')
print('  scp ftir_model.npz  pi@<PI_IP>:/home/pi/oil_health/models/')
print()
print('The pipeline will auto-load this on startup via FTIRModel.load()')
print('and use Model C coefficients for all TBN predictions from FTIR input.')

  ftir_model.npz              4228 bytes  ✓

Copy to Raspberry Pi:
  scp ftir_model.npz  pi@<PI_IP>:/home/pi/oil_health/models/

The pipeline will auto-load this on startup via FTIRModel.load()
and use Model C coefficients for all TBN predictions from FTIR input.


## How to run this notebook

1. Upload `daq_pipeline_ftir.py` as a Kaggle dataset (name it `daq-pipeline-ftir`)
2. Upload this notebook → **Run All**
3. Download `ftir_model.npz` from **Output → /kaggle/working/models/**
4. Copy to Pi: `scp ftir_model.npz pi@<PI_IP>:/home/pi/oil_health/models/`
5. The pipeline reads it automatically — no code changes needed

## Updating α corrections from your own data

Once you collect paired measurements `(band_abs, T, NTU, FD, TBN_lab)` from real engine runs:

```python
# Build extended feature matrix
X_ext = []
for row in your_data:
    bands = row['bands']          # dict of 5 band averages
    T, NTU, FD = row['T'], row['NTU'], row['FD']
    X_ext.append([
        bands['abs_lt1000'],    bands['abs_lt1000']    * (T-25), ...
        # etc — one column per band × covariate interaction
    ])
# Then call OfflineTrainer which refits FTIRModel.fit(X_ext, y_tbn)
python daq_pipeline_ftir.py --retrain
```